# Topic: SQL Percentile Calculation Pattern
***Note: MySQL Does not have PERCENTILE_CONT() and PERCENTILE_DISC() functions.*** 
## Definition (30-second explanation)
* A percentile tells you what percentage of values fall below a given value in a dataset.
* For example, if a student's score is at the 90th percentile, 90% of all students scored below them.
* SQL provides specialized window functions like `PERCENT_RANK()`, `CUME_DIST()`, `NTILE()`, and `PERCENTILE_CONT()` to compute these metrics efficiently without self-joins.

## Why Interviewers Ask This
* To test your ability to perform relative rankings and statistical bucketizations (like deciles and quartiles), which are foundational for Data Science tasks.
* To evaluate if you understand the subtle mathematical differences between cumulative distribution and relative positioning.
* To check if you know the specific syntax required for ordered-set aggregate functions (like median calculations).

## Core Concepts
* **PERCENT_RANK():** Calculates the relative position of a row, always ranging from 0.0 to 1.0 (the first row is always 0).
* **CUME_DIST():** Calculates the cumulative distribution, representing the fraction of rows less than or equal to the current row (never 0).
* **NTILE(n):** Divides the result set into `n` equal buckets, returning the bucket number (1 to n) for each row.
* **PERCENTILE_CONT(p) vs PERCENTILE_DISC(p):** `CONT` interpolates a continuous value that might not exist in the dataset (like an exact median), whereas `DISC` returns an actual discrete row value.

## When to Use
* **NTILE:** Use for cohort analysis or segmenting data into quartiles/deciles (e.g., categorizing top 25% of customers).
* **PERCENT_RANK:** Use for benchmarking a specific entity's relative position (e.g., where an employee's salary sits compared to peers).
* **PERCENTILE_CONT:** Use for finding precise statistical thresholds, most commonly the median (50th percentile) or interquartile range (IQR).
* **CUME_DIST:** Use when you need to know the total proportion of a population that falls at or below a certain threshold (e.g., Value at Risk in finance).

## Advantages
* Offloads heavy statistical calculations from Python/Pandas directly into the database engine, reducing data transfer size.
* Handles ties gracefully and mathematically correctly based on standard statistical formulas.

## Limitations
* `PERCENTILE_CONT` and `PERCENTILE_DISC` syntax (`WITHIN GROUP`) varies slightly across SQL dialects (e.g., PostgreSQL vs MySQL), and some older systems lack them entirely.
* Extreme outliers can skew `PERCENTILE_CONT` interpolations in very small datasets.

## Common Comparisons
* **PERCENT_RANK vs CUME_DIST for Ties:** If two rows tie, `PERCENT_RANK` gives them the same relative rank based on the start of the tie, whereas `CUME_DIST` calculates based on the maximum position of the tied rows.
* **PERCENTILE_CONT vs PERCENTILE_DISC:** `CONT` can generate a new number (e.g., averaging the two middle numbers for an even-count median), while `DISC` will pick one of the actual existing middle numbers.

## Common Interview Traps
* **Confusing PERCENT_RANK with CUME_DIST:** Remembering that `PERCENT_RANK` starts at 0 and `CUME_DIST` never hits 0 is a frequent interview follow-up.
* **Wrong ORDER BY direction:** `ORDER BY ASC` gives the fraction that is *less* than the current value; `DESC` gives the fraction that is *more*.
* **PERCENTILE_CONT Syntax:** Applying `OVER(PARTITION BY...)` to `PERCENTILE_CONT` instead of the required `WITHIN GROUP (ORDER BY ...)` syntax.
* **Forgetting to multiply by 100:** Both `PERCENT_RANK` and `CUME_DIST` return a decimal from 0.0 to 1.0; you often need to `* 100` and `ROUND()` for standard business readability.

## SQL Syntax 
```sql
-- Syntax for standard window functions
SELECT 
    PERCENT_RANK() OVER (PARTITION BY dept ORDER BY salary ASC) as pr,
    NTILE(4) OVER (ORDER BY salary DESC) as quartile
FROM employees;

-- Syntax for ordered-set aggregate functions (e.g., PostgreSQL/SQL Server)
SELECT 
    dept,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary ASC) as median_salary
FROM employees
GROUP BY dept;
```

## Important Formula
* `PERCENT_RANK()` = `(rank - 1) / (total_rows - 1)`.
* `CUME_DIST()` = `rank / total_rows`.

## 45-Second Interview Answer
"For percentile calculations in SQL, I rely on four main functions depending on the business need. I use NTILE() for simple bucketing like deciles or quartiles. If I need a relative percentile score for individual rows, I use PERCENT_RANK() or CUME_DIST(), being careful to multiply by 100. If I need to extract a specific statistical threshold from an aggregated group, like finding the exact median salary, I use PERCENTILE_CONT() combined with the WITHIN GROUP clause."

## Resume / Project Connection 
* **HR Analytics:** Salary benchmarking to determine which percentile each employee's salary falls into.
* **Marketing:** Customer Lifetime Value (LTV) percentile segmentation.
* **Finance:** Calculating Value at Risk (VaR) by identifying the 95th percentile of daily losses.

## Example Questions:

### Q1: Find all employees whose salary is above the 90th percentile in their department.
* **Ideal Answer:** 
```sql
WITH PercentileData AS (
    SELECT emp_id, emp_name, department, salary,
           PERCENT_RANK() OVER(PARTITION BY department ORDER BY salary ASC) as pct_rank
    FROM employees
)
SELECT * FROM PercentileData WHERE pct_rank > 0.90;
```
* **Common Mistakes:** Using `DESC` in the `ORDER BY` clause, which would inadvertently target the bottom 10% instead of the top 10%.
* **Follow-up:** "How would your logic change if we used `PERCENTILE_CONT(0.90)` instead of `PERCENT_RANK()`?" (Answer: I would calculate the 90th percentile threshold per department in a CTE using `PERCENTILE_CONT` and `GROUP BY`, then join it back to the main table to filter employees earning strictly more than that threshold).

### Q2: Classify all products into quartiles by revenue. Label them: 'Low', 'Medium', 'High', 'Premium'.
* **Ideal Answer:**
```sql
WITH Quartiles AS (
    SELECT product_id, revenue,
           NTILE(4) OVER(ORDER BY revenue ASC) as q_bucket
    FROM products
)
SELECT product_id, revenue,
       CASE 
           WHEN q_bucket = 1 THEN 'Low'
           WHEN q_bucket = 2 THEN 'Medium'
           WHEN q_bucket = 3 THEN 'High'
           WHEN q_bucket = 4 THEN 'Premium'
       END AS revenue_tier
FROM Quartiles;
```
* **Common Mistakes:** Forgetting that `NTILE(n)` requires an `ORDER BY` inside the `OVER()` clause to know how to sort the data before bucketing. 
* **Follow-up:** "What happens if there are 10 products and you ask for `NTILE(4)`?" (Answer: SQL attempts to distribute them evenly. The buckets will have sizes 3, 3, 2, 2. The remainder is distributed one by one to the top buckets).

### Q3: Calculate the interquartile range (IQR = P75 - P25) for student scores per class.
* **Ideal Answer:**
```sql
SELECT class_id,
       PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY score ASC) - 
       PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY score ASC) AS iqr
FROM student_scores
GROUP BY class_id;
```
* **Common Mistakes:** Trying to use `NTILE(4)` for this. `NTILE` only assigns bucket numbers; it does not calculate the actual boundary values (the 25th and 75th percentiles) needed for the IQR math.
* **Follow-up:** "Why is `PERCENTILE_CONT` preferred over `PERCENTILE_DISC` for this calculation?" (Answer: `PERCENTILE_CONT` interpolates the exact mathematical boundary for the 25th and 75th percentiles, providing a statistically accurate IQR, whereas `DISC` would just snap to the nearest student's actual score).

### Q4: Find the percentile rank of each customer by number of orders placed in the last 90 days.
* **Ideal Answer:**
```sql
SELECT customer_id, order_count_90d,
       ROUND(
           PERCENT_RANK() OVER(ORDER BY order_count_90d ASC) * 100, 
       2) AS order_percentile
FROM customer_activity;
```
* **Common Mistakes:** Using `CUME_DIST()` when the prompt specifically asks for "percentile rank", or forgetting to multiply by 100 to format it as a readable percentile (e.g., 85.50 instead of 0.855).
* **Follow-up:** "If five customers all have exactly 10 orders, how does `PERCENT_RANK()` treat their percentiles?" (Answer: They will all receive the exact same percentile rank, which corresponds to the lowest rank position of that tied group).

## Practice Questions:

### Q1: (The Classic MySQL Trap) Calculate the exact median `sale_price` per city without using PERCENTILE_CONT().

**Mock Schema**
```sql
-- Create the property_sales table
CREATE TABLE property_sales (
    property_id INT PRIMARY KEY,
    city VARCHAR(50),
    sale_price DECIMAL(12,2)
);

-- Insert sample data
INSERT INTO property_sales (property_id, city, sale_price) VALUES
-- Seattle: 3 sales (Odd count)
-- Ordered Prices: 500k, 600k, 5M (Outlier)
-- Median should be exactly 600,000.00
(1, 'Seattle', 500000.00),
(2, 'Seattle', 600000.00),
(3, 'Seattle', 5000000.00),

-- Portland: 4 sales (Even count)
-- Ordered Prices: 300k, 400k, 500k, 2M (Outlier)
-- Median should be (400k + 500k) / 2 = 450,000.00
(4, 'Portland', 300000.00),
(5, 'Portland', 500000.00),
(6, 'Portland', 400000.00),
(7, 'Portland', 2000000.00);
```
**Answer:**
```sql
WITH RankedSales AS (
    SELECT city, sale_price,
           ROW_NUMBER() OVER(PARTITION BY city ORDER BY sale_price ASC) as rnk,
           COUNT(*) OVER(PARTITION BY city) as total_rows
    FROM property_sales
)
SELECT city, 
       AVG(sale_price) AS median_price
FROM RankedSales
WHERE rnk BETWEEN (total_rows / 2.0) AND (total_rows / 2.0 + 1)
GROUP BY city;
```
* **Interview Tips:** Interviewers ask this specifically when testing for MySQL or SQLite roles because those dialects lack `PERCENTILE_CONT`. The trick is to generate a row number and a total count in a CTE, then filter for the middle row(s) using the `BETWEEN (total / 2.0) AND (total / 2.0 + 1)` formula. Finally, `GROUP BY` and `AVG()` those middle rows.
* **Common Mistakes:** Forgetting to divide by `2.0` (with the decimal). If you just divide by `2`, SQL might perform integer division (truncating the decimal), which breaks the mathematical boundaries for odd-numbered datasets.
* **Follow-up:** "If we were using PostgreSQL, how would you rewrite this query?" (Answer: I would use the built-in ordered-set aggregate function: `SELECT city, PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sale_price) FROM property_sales GROUP BY city;`).

### Q2:
**We need to benchmark our recent content. Write a SQL query to calculate two new metrics for each post within its respective network:**

- Calculate the PERCENT_RANK() of the engagement_score.

- Divide the posts into 3 equal performance tiers (terciles) based on their engagement_score. The highest scores should be in bucket 1.

- Return the post_id, network, engagement_score, and your two new calculated columns.

**Mock Schema**
```sql
CREATE TABLE social_media_posts (
    post_id VARCHAR(10),
    network VARCHAR(20),
    engagement_score INT
);

INSERT INTO social_media_posts (post_id, network, engagement_score) VALUES
-- Instagram Posts
('P1', 'Instagram', 1500),
('P2', 'Instagram', 1200),
('P3', 'Instagram', 900),
('P4', 'Instagram', 300),

-- LinkedIn Posts
('P5', 'LinkedIn', 400),
('P6', 'LinkedIn', 350),
('P7', 'LinkedIn', 100);
```


**Answer:**
```sql
SELECT *,
    ROUND(PERCENT_RANK() OVER(PARTITION BY network ORDER BY engagement_score ASC), 2) AS percentile,
    NTILE(3) OVER(PARTITION BY network ORDER BY engagement_score DESC) AS bucket
FROM social_media_posts;
```
* **Interview Tips:** Pay close attention to the `ORDER BY` direction inside your window functions. A single query might require `ASC` for one metric (like standard percentiles) and `DESC` for another (like "top tier" bucketing).
* **Common Mistakes:** Using `NTILE(3)` but forgetting to add `DESC`, which would accidentally put the worst-performing posts into bucket 1. 
* **Follow-up:** "If a network has 4 posts and you apply `NTILE(3)`, how does SQL distribute the extra post?" (Answer: It adds the remainder to the first bucket. The distribution will be 2 posts in bucket 1, 1 post in bucket 2, and 1 post in bucket 3).